# 04 — Dengue-Period Weather Aggregation and Dataset Preparation

**Project:** RiskWeave — Sri Lanka climate-informed dengue risk forecasting

## Purpose

This notebook converts daily ERA5 weather observations into weather summaries aligned with the actual dengue surveillance reporting periods, then combines them with dengue case counts.

## Important decisions for this project

1. **Actual reporting dates are used.** `week_start` and `week_end` are inclusive. A six-day period uses six daily weather records; an eight-day period uses eight. We do not force every period to seven days.
2. **25 weather districts are available.** The dengue source contains both `Ampara` and `Kalmune`, while the weather collection has only Ampara weather. Because the intended analysis unit is the 25 administrative districts, this notebook provides an explicit mapping that combines Kalmune dengue counts into Ampara. This is appropriate only if your project/supervisor confirms Kalmune is a reporting subdivision within Ampara District. The original dengue data is never changed.
3. **Both total and duration-normalized weather features are created.** For example, total precipitation and average daily precipitation are both retained. This helps avoid treating an eight-day accumulation as directly comparable to a six-day accumulation.
4. **Dengue cases remain counts.** They are summed across the Ampara and Kalmune reporting rows when the district-level mapping is applied. They are not divided by the number of days in this notebook.
5. **No source file is overwritten.** All outputs are new files under `data/processed/`.
6. **This is dataset preparation, not model training.** Forecast-origin timing, lag features, target construction and time-based validation will be designed in a later step to prevent leakage.

> **Geographic confirmation required:** The `Kalmune` → `Ampara` mapping below is an explicit project assumption based on your explanation that these are the same district geographically. Confirm the administrative/reporting relationship with an authoritative source or your supervisor before final academic analysis. If it is not approved, change the mapping to `None` and keep Kalmune outside the 25-district dataset.

## 1. Imports and project paths

This path setup works when the notebook is opened from the `notebooks` folder in VS Code or Jupyter. If your current working directory is already the project root, it will use that.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

WEATHER_PATH = (
    PROJECT_ROOT / "data" / "raw" / "weather_era5_2006_2025"
    / "weather_daily_era5_2006-12-23_to_2025-12-19.csv"
)
DENGUE_PATH = PROJECT_ROOT / "data" / "processed" / "dengue_weekly_cleaned.csv"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_WEATHER_PATH = OUTPUT_DIR / "weather_weekly_dengue_reporting_periods.csv"
MERGED_PATH = OUTPUT_DIR / "dengue_weather_weekly_25_districts.csv"
AUDIT_PATH = OUTPUT_DIR / "weather_period_aggregation_audit.csv"
DENGUE_DISTRICT_PATH = OUTPUT_DIR / "dengue_weekly_25_districts.csv"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Weather file exists:", WEATHER_PATH.exists())
print("Dengue file exists:", DENGUE_PATH.exists())

if not WEATHER_PATH.exists():
    raise FileNotFoundError(f"Weather file not found: {WEATHER_PATH}")
if not DENGUE_PATH.exists():
    raise FileNotFoundError(f"Dengue file not found: {DENGUE_PATH}")

## 2. Load the source datasets and inspect their columns

In [ ]:
weather = pd.read_csv(WEATHER_PATH)
dengue = pd.read_csv(DENGUE_PATH)

print("Daily weather shape:", weather.shape)
print("Weekly dengue shape:", dengue.shape)
print("\\nWeather columns:")
print(weather.columns.tolist())
print("\\nDengue columns:")
print(dengue.columns.tolist())

display(weather.head())
display(dengue.head())

## 3. Check required columns and convert data types

The weather source is expected to contain `district` and `date`. The dengue source is expected to contain `area`, `week_start`, `week_end`, and `dengue_cases`.

In [ ]:
required_weather = {
    "district", "date", "precipitation_sum", "precipitation_hours",
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "relative_humidity_2m_mean", "relative_humidity_2m_max",
    "relative_humidity_2m_min", "dew_point_2m_mean",
    "wind_speed_10m_mean", "wind_speed_10m_max",
    "wind_direction_10m_dominant", "cloud_cover_mean",
    "shortwave_radiation_sum", "et0_fao_evapotranspiration"
}
required_dengue = {"area", "week_start", "week_end", "dengue_cases"}

missing_weather = sorted(required_weather - set(weather.columns))
missing_dengue = sorted(required_dengue - set(dengue.columns))

if missing_weather:
    raise KeyError(f"Weather file is missing columns: {missing_weather}")
if missing_dengue:
    raise KeyError(f"Dengue file is missing columns: {missing_dengue}")

weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
dengue["week_start"] = pd.to_datetime(dengue["week_start"], errors="coerce")
dengue["week_end"] = pd.to_datetime(dengue["week_end"], errors="coerce")
dengue["dengue_cases"] = pd.to_numeric(dengue["dengue_cases"], errors="coerce")

print("Invalid weather dates:", weather["date"].isna().sum())
print("Invalid dengue start dates:", dengue["week_start"].isna().sum())
print("Invalid dengue end dates:", dengue["week_end"].isna().sum())
print("Missing dengue case values:", dengue["dengue_cases"].isna().sum())

## 4. Normalize area names and map dengue reporting areas to the 25 districts

The mapping is applied only to a copy of the dengue data. It does not modify `dengue_weekly_cleaned.csv`.

- Spelling/format aliases are standardized.
- `Kalmune` is mapped to `Ampara` under the district-level assumption described at the top.
- Dengue case counts are then summed by district and exact reporting dates.

In [ ]:
def normalize_area_name(value):
    return (
        str(value).strip().lower()
        .replace(" ", "").replace("_", "").replace("-", "")
    )

weather["district_key"] = weather["district"].map(normalize_area_name)
dengue["area_key"] = dengue["area"].map(normalize_area_name)

# Known name/format aliases between the two source files.
area_aliases = {
    "hambanthota": "hambantota",
    "monaragala": "moneragala",
}

weather["district_key"] = weather["district_key"].replace(area_aliases)
dengue["area_key"] = dengue["area_key"].replace(area_aliases)

# Project-level aggregation assumption: Kalmune reporting cases are included
# in Ampara District totals. Change to None if this mapping is not approved.
KALMUNE_TO_AMPARA = True

if KALMUNE_TO_AMPARA:
    dengue["district_key"] = dengue["area_key"].replace({"kalmune": "ampara"})
else:
    dengue["district_key"] = dengue["area_key"]

weather_districts = set(weather["district_key"].dropna().unique())
dengue_districts = set(dengue["district_key"].dropna().unique())

print("Number of weather districts:", len(weather_districts))
print("Number of dengue district keys after mapping:", len(dengue_districts))
print("Dengue district keys without weather:", sorted(dengue_districts - weather_districts))
print("Weather districts without dengue:", sorted(weather_districts - dengue_districts))

## 5. Aggregate dengue cases to district-reporting-period level

If Ampara and Kalmune are separate rows for the same reporting dates, their case counts are added to form the Ampara district total. The reporting dates are preserved exactly.

The output of this section should have one row per district and reporting interval.

In [ ]:
if (dengue["week_start"] > dengue["week_end"]).any():
    raise ValueError("At least one dengue reporting period has week_start after week_end.")

dengue["reporting_period_days"] = (
    dengue["week_end"] - dengue["week_start"]
).dt.days + 1

# Retain only areas represented in the 25-district weather dataset.
dengue_25_source_rows = dengue[dengue["district_key"].isin(weather_districts)].copy()

dengue_district = (
    dengue_25_source_rows
    .groupby(["district_key", "week_start", "week_end"], as_index=False)
    .agg(
        dengue_cases=("dengue_cases", "sum"),
        reporting_period_days=("reporting_period_days", "first"),
        source_reporting_rows=("area", "count"),
        source_areas=("area", lambda x: " | ".join(sorted(set(x.astype(str)))))
    )
)

# Verify that all rows grouped into a district-period agree on period length.
period_length_check = (
    dengue_25_source_rows
    .groupby(["district_key", "week_start", "week_end"])["reporting_period_days"]
    .nunique()
)
if (period_length_check > 1).any():
    raise ValueError("Inconsistent reporting-period lengths found within a district-period.")

print("District-level dengue shape:", dengue_district.shape)
print("Districts represented:", dengue_district["district_key"].nunique())
print("Reporting-period length counts:")
display(dengue_district["reporting_period_days"].value_counts().sort_index())

print("Rows where multiple source reporting areas were combined:")
display(dengue_district.loc[dengue_district["source_reporting_rows"] > 1].head(20))

## 6. Build the unique reporting-period calendar

The period calendar comes from the dengue dataset, not from an assumed Monday–Sunday calendar. Both start and end dates are inclusive.

For example:
- 18 April to 25 April = 8 days.
- 24 May to 29 May = 6 days.

The notebook calculates duration from the actual dates.

In [ ]:
periods = (
    dengue_district[["week_start", "week_end", "reporting_period_days"]]
    .drop_duplicates()
    .sort_values(["week_start", "week_end"])
    .reset_index(drop=True)
)
periods["period_id"] = np.arange(len(periods), dtype=int)

calculated_days = (periods["week_end"] - periods["week_start"]).dt.days + 1
if not calculated_days.equals(periods["reporting_period_days"]):
    raise ValueError("Reporting period length does not match inclusive start/end dates.")

print("Unique reporting periods:", len(periods))
display(
    periods["reporting_period_days"]
    .value_counts().sort_index()
    .rename_axis("period_days").to_frame("number_of_periods")
)
display(periods.loc[periods["reporting_period_days"] != 7])

## 7. Assign daily weather observations to the exact dengue reporting periods

A half-open interval `[week_start, week_end + 1 day)` is used internally. This includes the `week_end` date. Overlapping reporting intervals stop the notebook so they can be investigated rather than silently double-counting weather.

In [ ]:
intervals = pd.IntervalIndex.from_arrays(
    periods["week_start"],
    periods["week_end"] + pd.Timedelta(days=1),
    closed="left"
)

if intervals.is_overlapping:
    raise ValueError(
        "Dengue reporting intervals overlap. Inspect the dengue reporting dates before aggregation."
    )

weather = weather.dropna(subset=["date"]).copy()
weather["period_id"] = intervals.get_indexer(weather["date"])

weather_in_periods = weather.loc[weather["period_id"] >= 0].copy()

print("Daily weather rows assigned to a dengue period:", len(weather_in_periods))
print("Daily weather rows outside dengue periods:", len(weather) - len(weather_in_periods))

## 8. Define weather aggregation rules

**Accumulated variables** are summed across the exact reporting period:
- precipitation
- precipitation hours
- shortwave radiation
- ET0

**Daily state variables** are averaged across the period:
- temperatures
- relative humidity
- dew point
- wind speed
- cloud cover

**Wind direction** uses a circular mean. For example, 359° and 1° should average to approximately 0°, not 180°.

Your QA notebook found `rain_sum` exactly identical to `precipitation_sum` for all daily records. Therefore, the modelling table uses one precipitation variable to avoid duplication. The raw file remains unchanged.

In [ ]:
sum_columns = [
    "precipitation_sum",
    "precipitation_hours",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

mean_columns = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "cloud_cover_mean",
]

direction_column = "wind_direction_10m_dominant"

required_aggregation_columns = (
    ["district", "district_key", "date", "period_id"]
    + sum_columns + mean_columns + [direction_column]
)
missing = [c for c in required_aggregation_columns if c not in weather_in_periods.columns]
if missing:
    raise KeyError(f"Missing weather aggregation columns: {missing}")

def circular_mean_degrees(series):
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    if len(values) == 0:
        return np.nan
    radians = np.deg2rad(values)
    sin_mean = np.mean(np.sin(radians))
    cos_mean = np.mean(np.cos(radians))
    if np.isclose(sin_mean, 0) and np.isclose(cos_mean, 0):
        return np.nan
    return float(np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360)

aggregation = {column: "sum" for column in sum_columns}
aggregation.update({column: "mean" for column in mean_columns})

weekly_weather = (
    weather_in_periods
    .groupby(["district_key", "period_id"], as_index=False)
    .agg(
        **{column: (column, func) for column, func in aggregation.items()},
        wind_direction_10m_dominant=(direction_column, circular_mean_degrees),
        weather_days_observed=("date", "nunique"),
        weather_first_date=("date", "min"),
        weather_last_date=("date", "max"),
    )
)

weekly_weather = weekly_weather.merge(
    periods[["period_id", "week_start", "week_end", "reporting_period_days"]],
    on="period_id",
    how="left",
    validate="many_to_one"
)
weekly_weather["weather_days_expected"] = weekly_weather["reporting_period_days"]
weekly_weather["weather_days_complete"] = (
    weekly_weather["weather_days_observed"] == weekly_weather["weather_days_expected"]
)

## 9. Create duration-normalized weather features

For accumulated variables, retain both:
- the total over the reporting period; and
- the average per day (`total / reporting_period_days`).

This is important because six-day, seven-day and eight-day totals are not directly comparable. The normalized features describe average daily intensity, while totals preserve cumulative exposure.

For daily mean variables (temperature, humidity, wind speed, etc.), the period mean is already a daily-average summary, so no additional division is needed.

In [ ]:
for column in sum_columns:
    normalized_name = column.replace("_sum", "_mean_daily")
    if normalized_name == column:
        normalized_name = f"{column}_mean_daily"
    weekly_weather[normalized_name] = (
        weekly_weather[column] / weekly_weather["reporting_period_days"]
    )

# More explicit aliases for accumulated variables whose names do not end in "_sum".
weekly_weather["precipitation_hours_mean_daily"] = (
    weekly_weather["precipitation_hours"] / weekly_weather["reporting_period_days"]
)
weekly_weather["et0_fao_evapotranspiration_mean_daily"] = (
    weekly_weather["et0_fao_evapotranspiration"] / weekly_weather["reporting_period_days"]
)

display(weekly_weather.head())

## 10. Audit completeness for every district and reporting interval

A district-period is complete only if the number of unique daily weather observations equals the actual inclusive period length. This check naturally expects 6 observations for a six-day period and 8 for an eight-day period.

In [ ]:
district_keys = sorted(weather["district_key"].dropna().unique())
district_period_grid = pd.MultiIndex.from_product(
    [district_keys, periods["period_id"]],
    names=["district_key", "period_id"]
).to_frame(index=False)

audit = district_period_grid.merge(
    periods[["period_id", "week_start", "week_end", "reporting_period_days"]],
    on="period_id",
    how="left",
    validate="many_to_one"
).merge(
    weekly_weather[
        ["district_key", "period_id", "weather_days_observed",
         "weather_first_date", "weather_last_date", "weather_days_complete"]
    ],
    on=["district_key", "period_id"],
    how="left",
    validate="one_to_one"
)

audit["weather_days_observed"] = audit["weather_days_observed"].fillna(0).astype(int)
audit["weather_days_expected"] = audit["reporting_period_days"]
audit["weather_days_complete"] = audit["weather_days_complete"].fillna(False)
audit["missing_weather_days"] = (
    audit["weather_days_expected"] - audit["weather_days_observed"]
)

print("Expected district-period combinations:", len(audit))
print("Complete combinations:", int(audit["weather_days_complete"].sum()))
print("Incomplete combinations:", int((~audit["weather_days_complete"]).sum()))
display(audit.loc[~audit["weather_days_complete"]].head(30))

## 11. Merge district-level dengue and period-level weather

The merge uses the district key and exact start/end dates. Since Kalmune is mapped to Ampara above, its dengue counts are included in Ampara's district total, while the weather comes from the Ampara district weather series.

In [ ]:
weather_features = (
    sum_columns
    + [column.replace("_sum", "_mean_daily") for column in sum_columns]
    + mean_columns
    + [direction_column]
)

# Remove any duplicate feature names while preserving order.
weather_features = list(dict.fromkeys(weather_features))

weekly_weather_merge = weekly_weather[
    ["district_key", "week_start", "week_end",
     "weather_days_observed", "weather_days_expected",
     "weather_days_complete"] + weather_features
].copy()

merged = dengue_district.merge(
    weekly_weather_merge,
    on=["district_key", "week_start", "week_end"],
    how="left",
    validate="one_to_one"
)

print("Combined dataset shape:", merged.shape)
print("District count:", merged["district_key"].nunique())
print("Rows without a weather aggregate:", merged["weather_days_observed"].isna().sum())
print("Rows with incomplete weather:", (~merged["weather_days_complete"].fillna(False)).sum())
display(merged.head())

## 12. Final validation checks

We check:
- one row per district and reporting period;
- 25 districts in the combined dataset;
- reporting-period lengths are preserved;
- missing/incomplete weather periods are visible;
- dengue counts are non-negative;
- no unexpected duplicate merge keys exist.

In [ ]:
duplicate_count = merged.duplicated(
    subset=["district_key", "week_start", "week_end"]
).sum()

print("Duplicate district-period keys:", duplicate_count)
print("District count:", merged["district_key"].nunique())
print("Rows:", len(merged))
print("Reporting period length distribution:")
display(merged["reporting_period_days"].value_counts(dropna=False).sort_index())

print("Negative dengue case rows:", (merged["dengue_cases"] < 0).sum())
print("Missing dengue case rows:", merged["dengue_cases"].isna().sum())

if duplicate_count:
    raise ValueError("Duplicate district-period keys found in merged dataset.")
if (merged["dengue_cases"] < 0).any():
    raise ValueError("Negative dengue case counts found. Investigate source data.")

if merged["weather_days_observed"].isna().any():
    print("WARNING: Some district-periods have no matching weather aggregate.")
if (~merged["weather_days_complete"].fillna(False)).any():
    print("WARNING: Some district-periods have incomplete daily weather coverage.")

## 13. Save outputs

The following new files are created under `data/processed/`:

1. `dengue_weekly_25_districts.csv` — dengue cases aggregated to the 25-district level.
2. `weather_weekly_dengue_reporting_periods.csv` — weather aggregated to each actual dengue reporting period, including total and daily-normalized accumulated variables.
3. `dengue_weather_weekly_25_districts.csv` — combined dengue-weather dataset for analysis and later modelling.
4. `weather_period_aggregation_audit.csv` — completeness audit for every district-period.

The raw daily weather file and original cleaned dengue file are not overwritten.

In [ ]:
dengue_district.to_csv(DENGUE_DISTRICT_PATH, index=False)
weekly_weather.to_csv(WEEKLY_WEATHER_PATH, index=False)
merged.to_csv(MERGED_PATH, index=False)
audit.to_csv(AUDIT_PATH, index=False)

print("Saved files:")
print(DENGUE_DISTRICT_PATH)
print(WEEKLY_WEATHER_PATH)
print(MERGED_PATH)
print(AUDIT_PATH)

## 14. Interpretation and next step

The combined CSV has one row per **district per dengue reporting period**. It contains:
- reporting start and end dates;
- reporting-period duration in days;
- district-level dengue case count;
- accumulated weather totals;
- daily-normalized versions of accumulated weather variables;
- period-average temperature, humidity, dew point, wind and cloud cover;
- weather completeness indicators.

**Do not train the forecasting model directly from every column yet.** The next modelling step must define the forecast origin and next-period target carefully. In particular, future weather totals or the actual duration of a future reporting period must not be used as predictors unless they would genuinely be known at prediction time. We will use time-based validation and construct lagged features to prevent data leakage.